In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# ========== Load Files ==========
df1 = pd.read_csv("Normal_data.csv")
df2 = pd.read_csv("metasploitable-2.csv")
df3 = pd.read_csv("OVS.csv")

# ========== Merge ==========
df = pd.concat([df1, df2, df3], axis=0).reset_index(drop=True)

# ========== Clean ==========
df = df.drop_duplicates()
df = df.dropna()

# ========== Encode Categorical ==========
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# ========== Identify Label Column ==========
# Change if your label column name is different
label_col = "Label"

X = df.drop(label_col, axis=1)
y = df[label_col]

# ========== Normalize ==========
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# ========== Feature Extraction: PCA to 15 ==========
pca = PCA(n_components=15)
X_pca = pca.fit_transform(X_scaled)

# ========== Train Test Split ==========
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print("Final Shapes:")
print("Train:", X_train.shape, "Test:", X_test.shape)


Final Shapes:
Train: (275110, 15) Test: (68778, 15)


In [2]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

inp = Input(shape=(X_train.shape[1],))   # 15 features
h1 = Dense(20, activation='relu')(inp)
h2 = Dense(12, activation='relu')(h1)
h3 = Dense(6, activation='relu')(h2)     # deep compressed features

d1 = Dense(12, activation='relu')(h3)
d2 = Dense(20, activation='relu')(d1)
out = Dense(X_train.shape[1], activation='linear')(d2)

ae_model = Model(inp, out)
ae_model.compile(optimizer='adam', loss='mse')


In [3]:
ae_model.fit(X_train, X_train, epochs=30, batch_size=64, validation_split=0.1)


Epoch 1/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 21s 4ms/step - loss: 0.0256 - val_loss: 0.0050
Epoch 2/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0045 - val_loss: 0.0034
Epoch 3/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0032 - val_loss: 0.0025
Epoch 4/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0024 - val_loss: 0.0020
Epoch 5/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0019 - val_loss: 0.0018
Epoch 6/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 7/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0016 - val_loss: 0.0015
Epoch 8/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 9/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step - loss: 0.0014 - val_loss: 0.0013
Epoch 10/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 4ms/step - loss: 0.0013 - val_loss: 0.0012
Epoch 11/30
3869/3869 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - loss: 0.0013 - val_loss: 0.0012
Epoch 12/30
3869/38

In [4]:
ae_encoder = Model(inputs=inp, outputs=h3)

X_train_ae = ae_encoder.predict(X_train)
X_test_ae = ae_encoder.predict(X_test)

print("AE Feature Shape:", X_train_ae.shape)   # (samples, 6)


8598/8598 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step
2150/2150 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
AE Feature Shape: (275110, 6)


In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

hgb = HistGradientBoostingClassifier(max_depth=6)
hgb.fit(X_train_ae, y_train)


HistGradientBoostingClassifier(max_depth=6)

In [6]:
y_pred4 = hgb.predict(X_test_ae)

print("Hybrid Model 4: Deep AE + Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred4))
print(classification_report(y_test, y_pred4))


Hybrid Model 4: Deep AE + Boosting
Accuracy: 0.9941987263369101
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       281
           1       1.00      1.00      1.00        33
           2       1.00      1.00      1.00     14706
           3       1.00      1.00      1.00      9683
           4       0.99      0.99      0.99     10723
           5       0.99      1.00      0.99     13685
           6       1.00      0.99      0.99     19626
           7       0.02      0.67      0.04         3
           8       0.60      0.84      0.70        38

    accuracy                           0.99     68778
   macro avg       0.84      0.94      0.85     68778
weighted avg       1.00      0.99      0.99     68778

